# Open-source climate data — IMD gridded exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anantrajj7-sketch/lecture-deck/blob/main/imd_gridded_clip_exercise.ipynb)

Downloading IMD gridded rainfall data with `imdlib`, opening it as an `xarray` grid, and clipping it to a study area with a 33 km buffer around a shapefile.

Run each cell top to bottom. Cells you need to edit for your own study area are marked **EDIT ME**.


## 1. Install the libraries

Nothing is pre-installed in a fresh Colab runtime — this cell pulls in everything we need. Only needs to run once per session.


In [ ]:
!pip install -q imdlib geopandas shapely


## 2. Download the IMD gridded data

We'll pull all three variables IMD provides -- rainfall, min temperature, and max temperature -- across the same year range, since the goal is one GeoPackage with all three inside. **EDIT ME**: change the year range for your own use case.


In [ ]:
import imdlib as imd
import numpy as np

VARIABLES = ['rain', 'tmin', 'tmax']
start_year = 2020   # EDIT ME
end_year = 2023     # EDIT ME

FILL_VALUES = {'rain': -999, 'tmax': 99.9, 'tmin': 99.9}   # IMD's no-data value differs by variable

datasets = {}
for variable in VARIABLES:
    imd.get_data(variable, start_year, end_year, fn_format='yearwise')
    data = imd.open_data(variable, start_year, end_year, 'yearwise')
    ds = data.get_xarray()
    ds[variable] = ds[variable].where(~np.isclose(ds[variable], FILL_VALUES[variable]))
    datasets[variable] = ds

datasets['rain']


Before moving on: IMD's grid marks cells with no data (mostly ocean, outside India's landmass) with a fill number instead of `NaN` — **-999 for rainfall**, but **99.9 for temperature** (verified against real downloads of both). `get_xarray()` does **not** convert either one for you, so the loop above masks each variable by its own fill value. Skipping this would silently wreck an average later, and differently for temperature than for rainfall.


## 3. Recall: pulling a single point

This is the part you've already done before -- picking one lat/lon out of the grid. Shown here for rainfall; the same .sel() works on any of the three datasets.


In [ ]:
# EDIT ME: your point of interest
point_lat, point_lon = 28.6, 77.2   # Delhi, as an example

point_series = datasets['rain']['rain'].sel(lat=point_lat, lon=point_lon, method='nearest')
point_series.to_dataframe().reset_index().head()


## 4. Load your shapefile

**Option A (recommended for this class)** -- pick your Maharashtra district by name, it downloads automatically, no upload needed.

**Option B** -- upload your own zipped shapefile instead, if you're working with a different area.


In [ ]:
import geopandas as gpd

# All 36 Maharashtra districts (spelling matches the GeoPackage exactly):
#   AHAMADNAGAR, AKOLA, AMARAVATI, AURANGABAD
#   BHANDARA, BID, BULDHANA, CHANDRAPUR
#   DHULE, GADCHIROLI, GONDIA, HINGOLI
#   JALGAON, JALNA, KOLHAPUR, LATUR
#   MUMBAI CITY, NAGPUR, NANDED, NANDURBAR
#   NASHIK, PALGHAR, PARBHANI, PUNE
#   RATNAGIRI, RAYGAD, SANGLI, SATARA
#   SINDHUDURG, SOLAPUR, SUB URBAN MUMBAI, THANE
#   USMANABAD, WARDHA, WASHIM, YAVATMAL
DISTRICT = 'PUNE'   # EDIT ME -- your Maharashtra district, spelled exactly as in the list above

GPKG_URL = 'https://raw.githubusercontent.com/anantrajj7-sketch/lecture-deck/main/static/Data/maharashtra_districts.gpkg'
!wget -q {GPKG_URL} -O maharashtra_districts.gpkg

all_districts = gpd.read_file('maharashtra_districts.gpkg', layer='districts')
gdf = all_districts[all_districts['District'].str.upper().str.strip() == DISTRICT.upper()]

print('Found', len(gdf), 'match(es) for', DISTRICT)
gdf.plot()


### Option B: upload your own shapefile instead

Skip the cell above and run these two instead if you're working outside Maharashtra. Upload a **zipped** shapefile -- a `.zip` containing the `.shp`, `.shx`, `.dbf`, and (ideally) `.prj` files together. Zipping avoids having to upload four separate files one at a time.


In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # choose your shapefile .zip
zip_name = next(iter(uploaded))

extract_dir = 'shapefile'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

shp_files = [f for f in os.listdir(extract_dir) if f.endswith('.shp')]
shp_path = os.path.join(extract_dir, shp_files[0])
print('Using shapefile:', shp_path)


In [ ]:
import geopandas as gpd

gdf = gpd.read_file(shp_path).to_crs(epsg=4326)
gdf.plot()


## 5. Buffer the shapefile by 33 km

Buffering in degrees is wrong — a degree of longitude isn't a fixed distance, so the buffer would be a different real-world size depending on latitude. `estimate_utm_crs()` picks the right metric (meters-based) projection automatically, so we can buffer in meters and reproject back.


In [ ]:
BUFFER_METERS = 33_000   # EDIT ME if you need a different buffer distance

utm_crs = gdf.estimate_utm_crs()
buffered_metric = gdf.to_crs(utm_crs).buffer(BUFFER_METERS)
boundary = gpd.GeoSeries(buffered_metric, crs=utm_crs).to_crs(epsg=4326).union_all()
boundary


## 6. Clip every variable's grid to the buffer

Rainfall is a 0.25 degree grid, but temperature is 1 degree -- different shapes, different coordinates. So this repeats the point-in-polygon clip once per variable rather than reusing a single grid.


In [ ]:
import xarray as xr
from shapely.geometry import Point

clipped_gdfs = {}

for variable, ds in datasets.items():
    lat_vals = ds.lat.values
    lon_vals = ds.lon.values
    lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

    grid_points = gpd.GeoDataFrame(
        geometry=[Point(x, y) for x, y in zip(lon2d.ravel(), lat2d.ravel())],
        crs='EPSG:4326',
    )
    inside_2d = grid_points.within(boundary).values.reshape(lat2d.shape)
    mask = xr.DataArray(inside_2d, dims=['lat', 'lon'], coords={'lat': lat_vals, 'lon': lon_vals})

    clipped = ds[variable].where(mask)
    points_df = clipped.to_dataframe().reset_index().dropna(subset=[variable])
    points_gdf = gpd.GeoDataFrame(
        points_df,
        geometry=[Point(lon, lat) for lon, lat in zip(points_df['lon'], points_df['lat'])],
        crs='EPSG:4326',
    )
    clipped_gdfs[variable] = points_gdf
    print(variable, '-', len(points_gdf), 'rows kept')


## 7. Visual check

Plot the shapefile, the 33 km buffer, and the surviving rainfall points together -- a quick way to confirm the clip did what you expect before trusting the numbers.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
gpd.GeoSeries([boundary], crs='EPSG:4326').boundary.plot(ax=ax, color='tab:orange', linewidth=2, label='33 km buffer')
gdf.boundary.plot(ax=ax, color='black', linewidth=1, label='shapefile')
rain_points = clipped_gdfs['rain']
ax.scatter(rain_points['lon'], rain_points['lat'], s=8, color='tab:blue', label='kept grid points')
ax.legend()
ax.set_title('Rainfall grid points retained after clipping')
plt.show()


## 8. Save each variable as a CSV

One row per (time, lat, lon) that survived the clip, for a quick look in a spreadsheet. Fill values are already gone -- dropped in step 6 along with the rest of the NaNs.


In [ ]:
for variable, points_gdf in clipped_gdfs.items():
    csv_path = f'clipped_{variable}.csv'   # EDIT ME if you'd like different filenames
    points_gdf.drop(columns='geometry').to_csv(csv_path, index=False)
    print('Saved', len(points_gdf), 'rows to', csv_path)


## 9. Save all three into one GeoPackage

A GeoPackage can hold multiple tables in a single file — so rain, tmin, and tmax each get their own layer here, instead of three separate files. Opening clipped_points.gpkg in QGIS shows all three as separate layers to toggle on and off.


In [ ]:
GPKG_PATH = 'clipped_points.gpkg'   # EDIT ME if you'd like a different filename

for variable, points_gdf in clipped_gdfs.items():
    points_gdf.to_file(GPKG_PATH, layer=variable, driver='GPKG')

print('Saved layers:', list(clipped_gdfs.keys()), 'to', GPKG_PATH)

from google.colab import files
files.download(GPKG_PATH)   # prompts your browser to save it locally


## Recap

- Point vs. gridded data -- you pulled both a single point (step 3) and clipped a full region (steps 4-6) from the same downloaded grids.
- The grids were never "just your basin" -- imdlib always downloads the whole country; clipping is something *you* do afterward.
- Buffering must happen in a metric CRS, not directly in degrees.
- IMD's fill value is never `NaN` -- it's -999 for rainfall but 99.9 for temperature -- always mask it, by variable, before averaging.
- Rainfall (0.25 degree) and temperature (1 degree) grids have different shapes, so clipping is repeated per variable, not done once.
- CSV is fine for a quick look, but the GeoPackage (step 9) keeps geometry, CRS, and all three variables as separate layers in one file -- the right format to hand to a GIS tool.
